In [368]:
import pandas as pd
import numpy as np

In [369]:
manual_scores = pd.read_csv("/users/1/lundq163/projects/automated-qc/data/raw/auto_qc_t1w_t2w_subset_1024r_fixed.csv")
manual_scores.head()

,subject_id,session_id,run_id,suffix,scan,QU_motion,corrected_QU_motion
0,sub-100079,ses-V02,1,T2w,sub-100079_ses-V02_run-1_T2w.nii.gz,3.0,3.0
1,sub-100247,ses-V02,1,T2w,sub-100247_ses-V02_run-1_T2w.nii.gz,3.0,3.0
2,sub-100524,ses-V02,1,T2w,sub-100524_ses-V02_run-1_T2w.nii.gz,1.0,1.0
3,sub-100586,ses-V02,1,T2w,sub-100586_ses-V02_run-1_T2w.nii.gz,0.0,0.5
4,sub-100895,ses-V02,1,T2w,sub-100895_ses-V02_run-1_T2w.nii.gz,1.0,1.0


In [370]:
# drop the following columns: "subject_id", "session_id", "run_id", "suffix"
manual_scores = manual_scores.drop(columns=["subject_id", "session_id", "run_id", "suffix"])
manual_scores.head()

,scan,QU_motion,corrected_QU_motion
0,sub-100079_ses-V02_run-1_T2w.nii.gz,3.0,3.0
1,sub-100247_ses-V02_run-1_T2w.nii.gz,3.0,3.0
2,sub-100524_ses-V02_run-1_T2w.nii.gz,1.0,1.0
3,sub-100586_ses-V02_run-1_T2w.nii.gz,0.0,0.5
4,sub-100895_ses-V02_run-1_T2w.nii.gz,1.0,1.0


In [371]:
# rename the columns to be more descriptive
manual_scores = manual_scores.rename(columns={"QU_motion": "dons_scores", "corrected_QU_motion": "fj_updated_scores"})
manual_scores.head()

,scan,dons_scores,fj_updated_scores
0,sub-100079_ses-V02_run-1_T2w.nii.gz,3.0,3.0
1,sub-100247_ses-V02_run-1_T2w.nii.gz,3.0,3.0
2,sub-100524_ses-V02_run-1_T2w.nii.gz,1.0,1.0
3,sub-100586_ses-V02_run-1_T2w.nii.gz,0.0,0.5
4,sub-100895_ses-V02_run-1_T2w.nii.gz,1.0,1.0


In [372]:
model_scores = pd.read_csv("~/projects/automated-qc/doc/models/model_02r/model_02r5.csv")
model_scores.head()

,subject_id,session_id,run_id,suffix,scan,QU_motion,predicted_qu_motion_score
0,sub-100079,ses-V02,1,T2w,sub-100079_ses-V02_run-1_T2w.nii.gz,3.0,NaN
1,sub-100247,ses-V02,1,T2w,sub-100247_ses-V02_run-1_T2w.nii.gz,3.0,NaN
2,sub-100524,ses-V02,1,T2w,sub-100524_ses-V02_run-1_T2w.nii.gz,1.0,NaN
3,sub-100586,ses-V02,1,T2w,sub-100586_ses-V02_run-1_T2w.nii.gz,0.5,NaN
4,sub-100895,ses-V02,1,T2w,sub-100895_ses-V02_run-1_T2w.nii.gz,1.0,NaN


In [373]:
# drop the following columns: "subject_id", "session_id", "run_id", "suffix", "QU_motion"
model_scores = model_scores.drop(columns=["subject_id", "session_id", "run_id", "suffix", "QU_motion"])
model_scores.head()

,scan,predicted_qu_motion_score
0,sub-100079_ses-V02_run-1_T2w.nii.gz,NaN
1,sub-100247_ses-V02_run-1_T2w.nii.gz,NaN
2,sub-100524_ses-V02_run-1_T2w.nii.gz,NaN
3,sub-100586_ses-V02_run-1_T2w.nii.gz,NaN
4,sub-100895_ses-V02_run-1_T2w.nii.gz,NaN


In [374]:
# drop rows where predicted_qu_motion_score is NaN
model_scores = model_scores.dropna(subset=["predicted_qu_motion_score"])
model_scores.head()

,scan,predicted_qu_motion_score
12,sub-102673_ses-V02_run-1_T2w.nii.gz,0.762694
13,sub-102673_ses-V03_run-1_T2w.nii.gz,0.960739
24,sub-103199_ses-V02_run-1_T2w.nii.gz,0.674869
25,sub-103450_ses-V02_run-1_T2w.nii.gz,2.501921
26,sub-103450_ses-V02_run-2_T2w.nii.gz,2.443563


In [375]:
# merge the two dataframes on the scan column
merged_scores = pd.merge(manual_scores, model_scores, on="scan")
merged_scores.head()

,scan,dons_scores,fj_updated_scores,predicted_qu_motion_score
0,sub-102673_ses-V02_run-1_T2w.nii.gz,1.0,0.5,0.762694
1,sub-102673_ses-V03_run-1_T2w.nii.gz,0.0,0.5,0.960739
2,sub-103199_ses-V02_run-1_T2w.nii.gz,0.0,0.0,0.674869
3,sub-103450_ses-V02_run-1_T2w.nii.gz,1.0,2.0,2.501921
4,sub-103450_ses-V02_run-2_T2w.nii.gz,1.0,2.0,2.443563


In [376]:
merged_scores.shape

(213, 4)

In [377]:
thresholds = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]

for threshold in thresholds:
    # create binary labels for the manual scores using the threshold
    merged_scores[f"old_manual_label_{threshold}"] = merged_scores["dons_scores"] >= threshold
    merged_scores[f"manual_label_{threshold}"] = merged_scores["fj_updated_scores"] >= threshold
    # create binary labels for the model scores using the threshold
    merged_scores[f"model_label_{threshold}"] = merged_scores["predicted_qu_motion_score"] >= threshold

merged_scores


,scan,dons_scores,fj_updated_scores,predicted_qu_motion_score,old_manual_label_0.0,manual_label_0.0,model_label_0.0,old_manual_label_0.5,manual_label_0.5,model_label_0.5,...,model_label_1.5,old_manual_label_2.0,manual_label_2.0,model_label_2.0,old_manual_label_2.5,manual_label_2.5,model_label_2.5,old_manual_label_3.0,manual_label_3.0,model_label_3.0
0,sub-102673_ses-V02_run-1_T2w.nii.gz,1.0,0.5,0.762694,True,True,True,True,True,True,...,False,False,False,False,False,False,False,False,False,False
1,sub-102673_ses-V03_run-1_T2w.nii.gz,0.0,0.5,0.960739,True,True,True,False,True,True,...,False,False,False,False,False,False,False,False,False,False
2,sub-103199_ses-V02_run-1_T2w.nii.gz,0.0,0.0,0.674869,True,True,True,False,False,True,...,False,False,False,False,False,False,False,False,False,False
3,sub-103450_ses-V02_run-1_T2w.nii.gz,1.0,2.0,2.501921,True,True,True,True,True,True,...,True,False,True,True,False,False,True,False,False,False
4,sub-103450_ses-V02_run-2_T2w.nii.gz,1.0,2.0,2.443563,True,True,True,True,True,True,...,True,False,True,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208,sub-202422_ses-V06_run-1_T2w.nii.gz,0.0,0.5,0.677765,True,True,True,False,True,True,...,False,False,False,False,False,False,False,False,False,False
209,sub-204294_ses-V02_run-1_T2w.nii.gz,0.0,0.5,1.220863,True,True,True,False,True,True,...,False,False,False,False,False,False,False,False,False,False
210,sub-204294_ses-V02_run-1_T1w.nii.gz,1.0,2.5,2.183428,True,True,True,True,True,True,...,True,False,True,True,False,True,False,False,False,False
211,sub-204294_ses-V03_run-1_T2w.nii.gz,1.5,3.0,2.700174,True,True,True,True,True,True,...,True,False,True,True,False,True,True,False,True,False


In [378]:
merged_scores_old_manual_v_new_manual = merged_scores[[f"old_manual_label_{threshold}" for threshold in thresholds] + [f"manual_label_{threshold}" for threshold in thresholds]]
merged_scores_old_manual_v_new_manual.head()

,old_manual_label_0.0,old_manual_label_0.5,old_manual_label_1.0,old_manual_label_1.5,old_manual_label_2.0,old_manual_label_2.5,old_manual_label_3.0,manual_label_0.0,manual_label_0.5,manual_label_1.0,manual_label_1.5,manual_label_2.0,manual_label_2.5,manual_label_3.0
0,True,True,True,False,False,False,False,True,True,False,False,False,False,False
1,True,False,False,False,False,False,False,True,True,False,False,False,False,False
2,True,False,False,False,False,False,False,True,False,False,False,False,False,False
3,True,True,True,False,False,False,False,True,True,True,True,True,False,False
4,True,True,True,False,False,False,False,True,True,True,True,True,False,False


In [379]:
merged_scores_model_v_new_manual = merged_scores[[f"model_label_{threshold}" for threshold in thresholds] + [f"manual_label_{threshold}" for threshold in thresholds]]
merged_scores_model_v_new_manual.head()

,model_label_0.0,model_label_0.5,model_label_1.0,model_label_1.5,model_label_2.0,model_label_2.5,model_label_3.0,manual_label_0.0,manual_label_0.5,manual_label_1.0,manual_label_1.5,manual_label_2.0,manual_label_2.5,manual_label_3.0
0,True,True,False,False,False,False,False,True,True,False,False,False,False,False
1,True,True,False,False,False,False,False,True,True,False,False,False,False,False
2,True,True,False,False,False,False,False,True,False,False,False,False,False,False
3,True,True,True,True,True,True,False,True,True,True,True,True,False,False
4,True,True,True,True,True,False,False,True,True,True,True,True,False,False


In [380]:
analysis_dfs = [merged_scores_old_manual_v_new_manual, merged_scores_model_v_new_manual]
analysis_dfs

[     old_manual_label_0.0  old_manual_label_0.5  old_manual_label_1.0  \
 0                    True                  True                  True   
 1                    True                 False                 False   
 2                    True                 False                 False   
 3                    True                  True                  True   
 4                    True                  True                  True   
 ..                    ...                   ...                   ...   
 208                  True                 False                 False   
 209                  True                 False                 False   
 210                  True                  True                  True   
 211                  True                  True                  True   
 212                  True                 False                 False   
 
      old_manual_label_1.5  old_manual_label_2.0  old_manual_label_2.5  \
 0                   False          

In [381]:
markdown_content = "# Comparison of Model Predictions and Old Manual Labels to New Manual Labels\n\n"
markdown_content += "| y_pred | Threshold | Totals| TP | FP | TN | FN | Accuracy | PPV | NPV |\n"
markdown_content += "| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |\n"

for df in analysis_dfs:
    print(df.head())
    print(df.columns)
    
    metrics = []

    for threshold in thresholds:
        prediction_column = f"model_label_{threshold}" if "model_label" in df.columns[0] else f"old_manual_label_{threshold}"
        print(f"Analyzing {prediction_column} compared to new manual labels...")
        
        predicted = df[prediction_column]
        print(f"Predicted labels for threshold {threshold}:\n{predicted.head()}")
        actual = df[f"manual_label_{threshold}"]
        print(f"Actual labels for threshold {threshold}:\n{actual.head()}")
        tp = np.sum((predicted == True) & (actual == True))
        fp = np.sum((predicted == True) & (actual == False))
        tn = np.sum((predicted == False) & (actual == False))
        fn = np.sum((predicted == False) & (actual == True))

        accuracy = (tp + tn) / (tp + fp + tn + fn) if (tp + fp + tn + fn) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0

        markdown_content += f"| {prediction_column} | {threshold} | {tp + fp + tn + fn} | {tp} | {fp} | {tn} | {fn} | {accuracy:.4f} | {ppv:.4f} | {npv:.4f} |\n"
        
        metrics.append({
            'y_pred': prediction_column,
            'threshold': threshold,
            'accuracy': accuracy,
            'ppv': ppv,
            'npv': npv
        })

   old_manual_label_0.0  old_manual_label_0.5  old_manual_label_1.0  \
0                  True                  True                  True   
1                  True                 False                 False   
2                  True                 False                 False   
3                  True                  True                  True   
4                  True                  True                  True   

   old_manual_label_1.5  old_manual_label_2.0  old_manual_label_2.5  \
0                 False                 False                 False   
1                 False                 False                 False   
2                 False                 False                 False   
3                 False                 False                 False   
4                 False                 False                 False   

   old_manual_label_3.0  manual_label_0.0  manual_label_0.5  manual_label_1.0  \
0                 False              True              True      

In [382]:
output_md = f"comparison_to_new_manual_labels.md"

with open(output_md, 'w') as f:
    f.write(markdown_content)

In [51]:
# # create a table of accuracy, ppv, and npv comparing dons_scores and predicted_qu_motion_score to fj_updated_scores at each score threshold: 0.5, 1.0, 1.5, 2.0, 2.5, 3.0

# thresholds = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
# results = []
# for threshold in thresholds:
#     # create binary labels for the manual scores using the threshold
#     merged_scores["manual_label"] = merged_scores["fj_updated_scores"] >= threshold
#     print(merged_scores[["fj_updated_scores", "manual_label"]].head())
#     # create binary labels for the model scores using the threshold
#     merged_scores["model_label"] = merged_scores["predicted_qu_motion_score"] >= threshold
#     print(merged_scores[["predicted_qu_motion_score", "model_label"]].head())
#     # calculate accuracy, ppv, and npv for the model scores compared to the manual scores
#     accuracy = accuracy_score(merged_scores["manual_label"], merged_scores["model_label"])
#     ppv = precision_score(merged_scores["manual_label"], merged_scores["model_label"])
#     npv = precision_score(merged_scores["manual_label"], merged_scores["model_label"], pos_label=0)
#     results.append({"threshold": threshold, "accuracy": accuracy, "ppv": ppv, "npv": npv})
# results_df = pd.DataFrame(results)
# results_df

